# Download the TinyStories dataset

In [ ]:
!mkdir -p data
!cd data

!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!cd ..


# Import Packages/Modules

In [ ]:
import os
import pickle
import regex as re
from typing import BinaryIO
from itertools import repeat
import multiprocessing as mp
from collections import defaultdict, Counter
from concurrent.futures import ProcessPoolExecutor
from typing import Iterator, Iterable
from copy import deepcopy

# Profiling modules
import pstats
from cProfile import Profile

# Finding Chunk Boundaries
- Forked from `assignment1-basics/cs336_basics/pretokenization_example.py`.

In [2]:

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))

# Pre-Tokenization Function

In [3]:
PATTERN = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

def pre_tokenization(
                    file_path: str, 
                    special_tokens: list[str],
                    begin: int, 
                    terminate: int
                ) -> dict[tuple[bytes, ...], int]:
    
    with open(file_path, 'rb') as f:
        f.seek(begin)
        chunk = f.read(terminate - begin).decode("utf-8", errors="ignore")
        count: dict[tuple[bytes, ...], int] = {}

        # special_token = "<|endoftext|>"
        special_pattern = "|".join([re.escape(token) for token in special_tokens])
        chunk = re.split(special_pattern, chunk)

        for chunk_split in chunk:
            for m in re.finditer(PATTERN, chunk_split):
                bytes_tuple = tuple(bytes([ch]) for ch in m.group(0).encode('utf-8'))
                count[bytes_tuple] = count.get(bytes_tuple, 0) + 1
    return count


# Multiprocessing ProcessPoolExecutor Function
- Distributes tasks to child processes and collects, aggregates the data.

In [4]:
def mp_regex(
            file_path: str, 
            special_tokens: list[str],
            start: list[int] , 
            end: list[int], 
            num_workers: int | None = os.cpu_count()
        ) -> dict[tuple[bytes, ...], int]:

    ctx = mp.get_context("fork")
    
    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        results: Iterator[dict[tuple[bytes, ...], int]] = executor.map(pre_tokenization, repeat(file_path), repeat(special_tokens), start, end)
        
    pre_token_counts: Counter[tuple[bytes, ...]] = Counter()
    for worker_dict in results:
        pre_token_counts.update(worker_dict)
    return dict(pre_token_counts)


# Finding adjacent pairs and their appearance count function

In [5]:
def find_pairs(pre_token_count: dict[tuple[bytes, ...], int]) -> tuple[dict[tuple[bytes, bytes], int], dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]]:
    
    pairs: dict[tuple[bytes, bytes], int] = defaultdict(int)
    reverse_pair: dict[tuple[bytes, bytes], set[tuple[bytes, ...]]] = defaultdict(set)

    for tpl, appear in pre_token_count.items():
        for i in range(len(tpl)-1):
            pairs[(tpl[i], tpl[i+1])] += appear
            reverse_pair[(tpl[i], tpl[i+1])].add(tpl)
    return pairs, reverse_pair

# Update Funtions
- `pair_diff`: Finds pairs within a word with its count weighted by the word's count
- `remove_word_from_pair`: Removes a word from the bracket of every pair that appears in the given word.
- `add_word_to_pair`: Adds a word to the bracket of every pair that appears in the given word.
- `global_delta_pair`: Updates the pairs dictionary based on the delta (old count i.e. before merge - new count i.e. after merge) of the pair.

In [6]:
def pair_diff(
            word: tuple[bytes, ...],
            pre_token_count: dict[tuple[bytes, ...], int]
        ) -> dict[tuple[bytes, bytes], int]:
    pair_dict: dict[tuple[bytes, bytes], int] = defaultdict(int)
    for i in range(len(word)-1):
        pair_dict[(word[i], word[i+1])] += pre_token_count[word]
    return pair_dict

def remove_word_from_pair(
                        word: tuple[bytes, ...], 
                        pair_dict: dict[tuple[bytes, bytes], int], 
                        reverse_pair: dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]
                        ) -> dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]:
    for p in pair_dict:
        reverse_pair[p].remove(word)
    return reverse_pair

def add_word_to_pair(
                    word: tuple[bytes, ...], 
                    pair_dict: dict[tuple[bytes, bytes], int], 
                    reverse_pair: dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]
                    ) -> dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]:
    for p in pair_dict:
        reverse_pair[p].add(word)
    return reverse_pair


def global_delta_pair(
                    old_pair_dict: dict[tuple[bytes, bytes], int], 
                    new_pair_dict: dict[tuple[bytes, bytes], int], 
                    global_pair_dict: dict[tuple[bytes, bytes], int],
                    reverse_pair: dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]
                    ) -> dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]:
    
    delta_pair: dict[tuple[bytes, bytes], int] = {}
    delta_pair.update(old_pair_dict)
    delta_pair.update(new_pair_dict)
    for dp in delta_pair:
        delta_pair[dp] = old_pair_dict.get(dp, 0) - new_pair_dict.get(dp, 0)
        if global_pair_dict.get(dp, 0) - delta_pair.get(dp, 0) == 0:
            del global_pair_dict[dp]
            del reverse_pair[dp]
        elif global_pair_dict.get(dp, 0) - delta_pair.get(dp, 0) > 0:
            global_pair_dict[dp] = global_pair_dict.get(dp, 0) - delta_pair.get(dp, 0)
    return reverse_pair



# Merge Function

In [7]:
def merge(
        pre_token_count: dict[tuple[bytes, ...], int], 
        pair: tuple[bytes, bytes], 
        reverse_pair: dict[tuple[bytes, bytes], set[tuple[bytes, ...]]],
        pairs: dict[tuple[bytes, bytes], int],
        verbose: bool = False
    ) -> tuple[dict[tuple[bytes, ...], int], dict[tuple[bytes, bytes], set[tuple[bytes, ...]]]]:
    
    total_merges: int = 0
    words: set[tuple[bytes, ...]] = deepcopy(reverse_pair[pair])

    for word in words:
        i = 0
        n = len(word)
        end_flag: bool = False
        merge_happen: bool = False
        merged_word: list[bytes] = []
        appear: int = pre_token_count[word]

        old_pair_dict = pair_diff(word, pre_token_count)
        reverse_pair = remove_word_from_pair(word, old_pair_dict, reverse_pair)

        while i < n-1:
            if (pair[0], pair[1]) == (word[i], word[i+1]):
                merged_word.append(pair[0]+pair[1])
                end_flag = True if i == n-2 else False
                merge_happen = True
                i += 2
                total_merges += 1
                continue
            merged_word.append(word[i])
            i += 1

        if not end_flag:
            merged_word.append(word[-1])

        new_tpl: tuple[bytes, ...] = tuple(merged_word)

        del pre_token_count[word]
        pre_token_count[new_tpl] = appear
        
        new_pair_dict = pair_diff(new_tpl, pre_token_count)
        reverse_pair = add_word_to_pair(new_tpl, new_pair_dict, reverse_pair)

        reverse_pair = global_delta_pair(old_pair_dict, new_pair_dict, global_pair_dict=pairs, reverse_pair=reverse_pair)

        if verbose and merge_happen:
            print(f"Merge Successful with {pair=} resulting in new string {new_tpl=}")

    return pre_token_count, reverse_pair

# Train Function
- Trains the tokenizer for a given number of steps.

In [ ]:
def train(
        input_path: str,
        vocab_size: int, 
        special_tokens: list[str]
    ) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    
    i2b_vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    merge_iters: int = vocab_size - (len(i2b_vocab) + len(special_tokens))        
    
    merge_order: list[tuple[bytes, bytes]] = []

    # Chunking file / finding chunk boundaries
    with open(input_path, "rb") as f:
        num_processes = 8
        boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

    start = boundaries[:-1]
    end = boundaries[1:]
    
    print(f"{start}\n{end}")

    # Parallelilzing pre-tokenization using lazy RegEx (re.finditer)
    pre_token_count: dict[tuple[bytes, ...], int] = mp_regex(
                                                            file_path=input_path, 
                                                            special_tokens=special_tokens, 
                                                            start=start, 
                                                            end=end, 
                                                            num_workers=num_processes
                                                            )
    print(len(pre_token_count))
    
    pairs, reverse_pair = find_pairs(pre_token_count)

    # Merge Steps
    for _ in range(merge_iters):
        max_pair: tuple[bytes, bytes] = max(pairs, key=lambda k: (pairs[k], k))

        v_idx: int = max(i2b_vocab) + 1
        b_string: bytes = max_pair[0] + max_pair[1]

        merge_order.append(max_pair)

        i2b_vocab[v_idx] = b_string
        # b2i_vocab[b_string] = v_idx
        
        pre_token_count, reverse_pair = merge(pre_token_count, pair=max_pair, reverse_pair=reverse_pair, pairs=pairs)

    # Appending special_tokens to the vocabulary
    for token in special_tokens:
        i2b_vocab[max(i2b_vocab)+1] = token.encode('utf-8')

    return i2b_vocab, merge_order

In [ ]:
target_path = "/media/nightking/WD-SN570/deep_learning/stanford_cs336/assignment1-basics/tests/fixtures/corpus.en"
special_tokens = ["<|endoftext|>"]


with Profile() as profile:
    i2b_vocab, merge_order = train(input_path=target_path, vocab_size=500, special_tokens=special_tokens)
profile.dump_stats("notebook_train.prof")


stats = pstats.Stats("notebook_train.prof")
stats.sort_stats("cumtime").print_stats(15)      # top 15 by cumulative time
stats.print_callers("find_pairs")                # who calls into find_pairs?

[0]
[133027]
4763
Mon Aug 31 19:36:08 2026    notebook_train.prof

         2419950 function calls (2297316 primitive calls) in 0.774 seconds

   Ordered by: cumulative time
   List reduced from 532 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      243    0.049    0.000    0.415    0.002 /tmp/ipykernel_324441/3671951420.py:1(merge)
        1    0.001    0.001    0.394    0.394 /tmp/ipykernel_324441/563657020.py:1(train)
    15404    0.106    0.000    0.208    0.000 /tmp/ipykernel_324441/3800591529.py:29(global_delta_pair)
      499    0.090    0.000    0.149    0.000 {built-in method builtins.max}
    40/39    0.000    0.000    0.114    0.003 {method 'run' of '_contextvars.Context' objects}
   936609    0.096    0.000    0.096    0.000 {method 'get' of 'dict' objects}
      2/1    0.000    0.000    0.093    0.093 /home/nightking/miniconda3/lib/python3.14/threading.py:1030(_bootstrap)
      2/1    0.000    0.000    0.093    0.09

# Vocab Inspection

In [ ]:
def save_checkpoint(vocab: dict[int, bytes], merges: list[tuple[bytes, bytes]]):
    save_vocab = {i: j.decode("latin-1") for i, j in vocab.items()}

    with open("vocab.json", "w", encoding="latin-1") as f1:
        json.dump(save_vocab, f1, indent=4)

    with open("merges.pkl", "wb") as f2:
        pickle.dump(merges, f2)

    print(f"Files saved....")

In [ ]:
print(f"{'Length of Vocab':<50}: {len(i2b_vocab)}\n")
print("-"*54)
for key, value in i2b_vocab.items():
    print(f"{'Index':<10}: {key:>5} {'|':^5} {'Byte_String':<12}: {repr(value):>15}")



# Tokenizer Class

In [ ]:
import json

class Tokenizer:
    def __init__(
            self, 
            vocab: dict[int, bytes], 
            merges: list[tuple[bytes, bytes]], 
            special_tokens: list[str] | None = None
            ):

        self.vocab = vocab
        self.merges = merges
        self.special_tokens = special_tokens
        self.reverse_vocab = {y: x for x, y in vocab.items()}
        self.merge_lookup = {y: x for x, y in enumerate(merges)}
        self.PATTERN = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
        self.stkns_2_id = {}

        if self.special_tokens:
            self.special_tokens.sort(reverse=True, key=lambda x: len(x))
            self.stkns_2_id = {token.encode('utf-8'): self.reverse_vocab[token.encode('utf-8')] for token in self.special_tokens}


    @classmethod
    def from_files(
            cls, 
            vocab_filepath: str,
            merges_filepath: str,
            special_tokens: list[str] | None = None
        ):

        # Load vocabulary dict
        with open(vocab_filepath, 'r') as file1:
            vocab = json.load(file1)

        vocab = {int(i): j.encode("latin-1") for i, j in vocab.items()}
        
        # Load merges order list
        with open(merges_filepath, 'rb') as file2:
            merges = pickle.load(file2)

        return cls(vocab, merges, special_tokens)



    def encode(self, text: str) -> list[int]:

        encoded_list: list[int] = []
        
        if self.special_tokens:
            special_pattern = "|".join([re.escape(token) for token in self.special_tokens])
            text: list[str] = re.split(f"""({special_pattern})""", text)
        else:
            text = [text]

        for chunk_split in text:
            if chunk_split == "":
                continue
            elif chunk_split.encode('utf-8') in self.stkns_2_id:
                encoded_list.append(self.stkns_2_id[chunk_split.encode('utf-8')])
                continue
            pre_tokens: list[tuple[bytes, ...]] = []
            for m in re.finditer(self.PATTERN, chunk_split):
                bytes_tuple = tuple(bytes([ch]) for ch in m.group(0).encode('utf-8'))
                pre_tokens.append(bytes_tuple)

            for w_idx, word in enumerate(pre_tokens):
                word_copy = list(word)

                for _ in range(len(word)-1):
                    n = len(word_copy)
                    min_rank: tuple[int | float, tuple[bytes, bytes]] = (float('inf'), (b"", b""))  #Placeholder bytes for type annotations

                    for i in range(n-1):
                        ml_pair: int | float = self.merge_lookup.get((word_copy[i], word_copy[i+1]), float('inf'))
                        min_rank = min(min_rank, (ml_pair, (word_copy[i], word_copy[i+1])))

                    if min_rank[0] == float('inf'):
                        break

                    for j in range(n-1, 0, -1):
                        if (word_copy[j-1], word_copy[j]) == min_rank[1]:
                            word_copy[j] = word_copy[j-1] + word_copy[j]
                            del word_copy[j-1]
                pre_tokens[w_idx] = tuple(word_copy)

            encoded_list += [self.reverse_vocab[x] for token in pre_tokens for x in token]

        return encoded_list


    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:

        for line_text in iterable:
            # encoded_list: list[int] = []
            
            if self.special_tokens:
                special_pattern = "|".join([re.escape(token) for token in self.special_tokens])
                text: list[str] = re.split(f"""({special_pattern})""", line_text)
            else:
                text = [line_text]

            for chunk_split in text:
                if chunk_split == "":
                    continue
                elif chunk_split.encode('utf-8') in self.stkns_2_id:
                    # encoded_list.append(self.stkns_2_id[chunk_split.encode('utf-8')])
                    yield self.stkns_2_id[chunk_split.encode('utf-8')]
                    continue
                pre_tokens: list[tuple[bytes, ...]] = []
                for m in re.finditer(self.PATTERN, chunk_split):
                    bytes_tuple = tuple(bytes([ch]) for ch in m.group(0).encode('utf-8'))
                    pre_tokens.append(bytes_tuple)

                for word in pre_tokens:
                    word_copy = list(word)

                    for _ in range(len(word)-1):
                        n = len(word_copy)
                        min_rank: tuple[int | float, tuple[bytes, bytes]] = (float('inf'), (b"", b""))  #Placeholder bytes for type annotations

                        for i in range(n-1):
                            ml_pair: int | float = self.merge_lookup.get((word_copy[i], word_copy[i+1]), float('inf'))
                            min_rank = min(min_rank, (ml_pair, (word_copy[i], word_copy[i+1])))

                        if min_rank[0] == float('inf'):
                            break

                        for j in range(n-1, 0, -1):
                            if (word_copy[j-1], word_copy[j]) == min_rank[1]:
                                word_copy[j] = word_copy[j-1] + word_copy[j]
                                del word_copy[j-1]
                    merged_word = tuple(word_copy)
                    # pre_tokens[w_idx] = merged_word

                    for token in merged_word:
                            yield self.reverse_vocab[token]


    def decode(self, ids: list[int]) -> str:
        byte_string = [self.vocab[i] for i in ids]
        return b"".join(byte_string).decode('utf-8', errors='replace')
